# ORE Neurocomputational — Cloud Test Runner

This notebook clones the public ORE repository and runs its existing pytest suite plus the randomized Workbench evaluation.

**Important:** ordinary pytest and the current randomized runner are CPU workloads. A GPU will not automatically make them faster. Select **Accelerator = None** for the default run; choose GPU only if the specific tests you are running use CUDA-enabled code.

The result archive is saved to `/kaggle/working/ORE-cloud-test-results.zip`.

In [ ]:
from pathlib import Path
import shutil, subprocess, sys

REPO_URL = "https://github.com/wizekin/ORE-Neurocomputational.git"
REPO_DIR = Path("/kaggle/working/ORE-Neurocomputational")
TRIALS_PER_CALLABLE = 5  # Increase to 25 for a larger evaluation
MAX_CALLABLES_PER_PACKAGE = 0  # 0 means all discovered eligible callables
RANDOM_SEED = None  # Set an integer to reproduce a randomized run

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".[dev]"], cwd=REPO_DIR, check=True)

command = [
    sys.executable, "cloud_testing/run_cloud_tests.py",
    "--trials-per-callable", str(TRIALS_PER_CALLABLE),
    "--max-callables-per-package", str(MAX_CALLABLES_PER_PACKAGE),
    "--output-dir", "/kaggle/working/ore_cloud_results",
]
if RANDOM_SEED is not None:
    command += ["--random-seed", str(RANDOM_SEED)]

completed = subprocess.run(command, cwd=REPO_DIR, check=False)
archive = next(Path("/kaggle/working/ore_cloud_results").rglob("ORE-cloud-test-results.zip"), None)
if archive is not None:
    shutil.copy2(archive, "/kaggle/working/ORE-cloud-test-results.zip")
    print("\nDownload this file from the Kaggle output panel: /kaggle/working/ORE-cloud-test-results.zip")
else:
    print("The runner did not produce an archive. Inspect the cell output for the error.")

print("Runner exit code:", completed.returncode)